In [10]:
import itertools
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window
from sklearn.cluster import KMeans
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV
import elapid as ela
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

In [12]:
FEATURES = ["elevation", "slope","aspect","hcas"]

In [18]:
DATA_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()

In [20]:
SPECIES_CONFIGS=[
    {
        "species_name":"amytornis_purnelli",
        "training_csv":DATA_DIR/"amytornis_purnelli_training_matrix.csv",
        "raster_paths": {
            "elevation": DATA_DIR/"amytornis_purnelli_elevation.tif",
            "slope":DATA_DIR/"amytornis_purnelli_slope.tif",
            "aspect":DATA_DIR/"amytornis_purnelli_aspect.tif",
            "hcas":DATA_DIR/"amytornis_purnelli_hcas.tif",
        },
    },
    {
        "species_name":"atrichornis_rufescens",
        "training_csv":DATA_DIR/"atrichornis_rufescens_training_matrix.csv",
        "raster_paths": {
            "elevation": DATA_DIR/"atrichornis_rufescens_elevation.tif",
            "slope":DATA_DIR/"atrichornis_rufescens_slope.tif",
            "aspect":DATA_DIR/"atrichornis_rufescens_aspect.tif",
            "hcas":DATA_DIR/"atrichornis_rufescens_hcas.tif",
        },
    },
    {
        "species_name":"pedionomous_torquatus",
        "training_csv":DATA_DIR/"pedionomous_torquatus_training_matrix.csv",
        "raster_paths": {
            "elevation": DATA_DIR/"pedionomous_torquatus_elevation.tif",
            "slope":DATA_DIR/"pedionomous_torquatus_slope.tif",
            "aspect":DATA_DIR/"pedionomous_torquatus_aspect.tif",
            "hcas":DATA_DIR/"pedionomous_torquatus_hcas.tif",
        },
    },
    {
        "species_name":"pezoporus_occidentalis",
        "training_csv":DATA_DIR/"pezoporus_occidentalis_training_matrix.csv",
        "raster_paths": {
            "elevation": DATA_DIR/"pezoporus_occidentalis_elevation.tif",
            "slope":DATA_DIR/"pezoporus_occidentalis_slope.tif",
            "aspect":DATA_DIR/"pezoporus_occidentalis_aspect.tif",
            "hcas":DATA_DIR/"pezoporus_occidentalis_hcas.tif",
        },
    },
    {
        "species_name":"polytelis_alexandrae",
        "training_csv":DATA_DIR/"polytelis_alexandrae_training_matrix.csv",
        "raster_paths": {
            "elevation": DATA_DIR/"polytelis_alexandrae_elevation.tif",
            "slope":DATA_DIR/"polytelis_alexandrae_slope.tif",
            "aspect":DATA_DIR/"polytelis_alexandrae_aspect.tif",
            "hcas":DATA_DIR/"polytelis_alexandrae_hcas.tif",
        },
    },
]     

In [22]:
N_SPATIAL_FOLDS = 5

In [24]:
RANDOM_STATE = 1234

In [26]:
OUTPUT_DIR = DATA_DIR/"outputs"

In [28]:
OUTPUT_DIR.mkdir(exist_ok=True)

# Step 1 - Loading the pre built training matrix

In [33]:
def sample_rasters_at_points(raster_paths: dict,coords: list[tuple[float,float]]) -> pd.DataFrame:
    out = {}
    for name,path in raster_paths.items():
        with rasterio.open(path) as src:
            out[name] = [val[0] for val in src.sample(coords)]
    return pd.DataFrame(out)

In [43]:
def load_training_matrix(csv_path: Path, features: list) -> pd.DataFrame: 
    df = pd.read_csv(csv_path)
    required = {"x_coord","y_coord","presence", * features}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{csv_path.name} is missing required columns: {missing}")
    df = df.dropna(subset = features)
    n_pos, n_neg = (df["presence"] == 1).sum(),(df["presence"] == 0).sum()
    print(f"Loaded {len(df)} rows ({n_pos} presence, {n_neg} background)")
    return df[["x_coord","y_coord","presence"] + features]

# Step 2 - Generate Background points from raster data

In [56]:
def generate_background_points(raster_paths:dict, n_points: int, random_state: int, block_rows: int = 512) -> pd.DataFrame:
    features = list(raster_paths.keys())
    rng = np.random.default_rng(random_state)
    ref_path = next(iter(raster_paths.values()))

    with rasterio.open(ref_path) as src:
        nodata = src.nodata
        height, width = src.heights, src.width
        n_blocks = (heigh + block_rows - 1) // block_rows

        #Counting valid pixels per block
        block_counts = np.zeros(n_blocks, dtype = np.int64)
        for bi, row_start in enumerate(range(0, height, block_rows)):
            rows = min(block_rows, height - row_start)
            window = Window(0,row_star, width, rows)
            band = src.read(1, window = window)
            valid_mask = np.isinfinite(band) if nodata is None else (band != nodata) & np.isfinite(band)
            block_counts[bi] = valid_mask.sum()
        total_valid = block_counts.sum()
        if total_valid ==0:
            raise ValueError(f"No Valid (non-nodata) pixels found in {ref_path}")
            if total_valid ==0:
                raise ValueError(f"No valid (non-nodata) pixels found in {ref-path}")
            if total_valid < n_points:
                print (f"Warning: only {total_valid} valid pixels available, requested {n_points}")
                n_points = int(total_valid)

            raw_alloc = block_counts/total_valid * n_points
            alloc = np.floor(raw_alloc).astype(np.int64)
            shortfall = n_points - alloc.sum()
            if shortfall > 0:
                remainders = raw_alloc - alloc
                top_up = np.argsort(-remainders)[:shortfall]
                alloc[top_up] += 1
            alloc = np.minimum(alloc, block_counts)

            # Vectorising sample within each block
            all_rows, all_cols -[], []
            for bi, row_start in enumerate(range(0,height, block_rows)):
                if alloc[bi]==0:
                    continue
                rows = min(block_rows, height - row_start)
                window = Window(0,row_start, width, rows)
                band = src.read(1, window=window)
                valid_mask = np.isfinite(band) if nodata is None else (band!=nodata) & np.isfinite(band)
                local_rows,local_cols = np.where(valid_mask)
                chosen=rng.choice(len(local_rows),size=int(alloc[bi],replace =False))
                all_rows.append(local_rows[chosen] +row_start)
                all_cols.appemd(local_cols[chosen])
            chosen_rows = np.concatenate(all_rows)
            chosen_cols = np.concatenate(all_cols)
            xs,ys = rasterio.transform.xy(src.transform,chosen_rows,chosen_cols)
            coords = list(zip(xs,ys))
            bg = sample_rasters_at_points(raster_paths,coords)
            bg["x_coord"] = xs
            bg["y_coord"] = ys
            bg["presence"] = 0
            bg=bg.dropna(subset=features)
            return bg[["x_coord","y_coord","presence"] + features]